# Fast movers

**Goal:** find objects that move so fast they leave a short **trail** in one 30-second exposure (mostly near-Earth objects),
and keep false alarms near zero. Read `docs/fast_movers.md` alongside this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from asthunt.config.load_config import load_config
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.acquire.select_night_exposures import select_night_exposures
from asthunt.clean.load_exposure import load_exposure
from asthunt.simulate.render_trailed_source import render_trailed_source
from asthunt.simulate.plan_injections import plan_injections
from asthunt.fast.build_streak_kernels import build_streak_kernels
from asthunt.fast.find_streaks import find_streaks
from asthunt.pipeline.run_fast_search import run_fast_search

project = load_config("project"); paths = project["paths"]
acquire, fast = load_config("acquire"), load_config("fast")
unit = {"field": 618, "ccd": 3, "quadrant": 4, "night": "2019-02-08"}
metadata = load_unit_metadata(unit, paths["raw"])

## 1. How long is a trail?
Distance moved during the exposure: `rate ("/hr) x 30 s / 3600 / 1.012 "/px` ≈ rate / 120 pixels.

In [ ]:
for rate in [120, 480, 1000, 2400, 3800]:
    print(f"{rate:5d} \"/hr -> trail of {rate * fast['exposure_seconds'] / 3600 / 1.012:5.1f} px")

frame = select_night_exposures(metadata, unit["night"], acquire["filter"], 1).iloc[0]
exposure = load_exposure(paths["raw"] / "ztf" / unit["night"] / f"ztf_{frame['filefracday']}_000618_zr_c03_o_q4_sciimg.fits", acquire["bad_mask_bits"])
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, length in zip(axes, [0, 4, 16, 32]):
    ax.imshow(render_trailed_source(exposure["psf"], 1.0, length, 30, size=45), origin="lower"); ax.set_title(f"{length} px trail"); ax.axis("off")
plt.show()

## 2. The kernel bank and the single-frame problem
Raw difference images give thousands of "streaks" per frame: long kernels integrate over halos and detector-aligned residuals.

In [ ]:
kernels = build_streak_kernels(exposure["psf"], [4, 8, 16, 32, 64], fast["angle_step_degrees"])
raw = find_streaks(exposure["difference"], exposure["bad"], exposure["psf"], kernels, fast["detection_threshold_sigma"])
print(f"{len(kernels)} kernels; raw difference image: {len(raw)} streak detections; by length {raw['length'].value_counts().sort_index().to_dict()}")

## 3. Real night vs planted trails, through the full fast search (standardised over the frames)
The real night gives the false alarms; the planted trails tell us what real objects look like.

In [ ]:
real = run_fast_search(unit, metadata, acquire, fast, paths, progress=lambda m: None)
frames = select_night_exposures(metadata, unit["night"], acquire["filter"], 5)
rng = np.random.default_rng(11)
fakes = plan_injections(150, (3080, 3072), frames["obsjd"].tolist(), (15.0, 20.0), (400.0, 5000.0), 1.012, 100, rng)
planted = run_fast_search(unit, metadata, acquire, fast, paths, fakes=fakes, rng=rng, progress=lambda m: None)
print(f"real night : {len(real['streaks'])} streaks, {len(real['tracklets'])} linked, consistent {int(real['tracklets']['consistent'].sum()) if len(real['tracklets']) else 0}")
print(f"with fakes : {len(planted['streaks'])} streaks, {len(planted['tracklets'])} linked, consistent {int(planted['tracklets']['consistent'].sum())}")

In [ ]:
def matched_fake(result, fakes, radius=6):
    """For each tracklet: the fake it follows in >= 3 frames, or -1."""
    streaks, members = result["streaks"], result["members"]
    answer = {}
    for tracklet_id, group in members.groupby("tracklet_id"):
        rows = streaks.loc[group["detection_id"]]
        for fake_id, track in fakes.groupby("injection_id"):
            hits = sum(((track["frame_index"] == r.frame_index) & (np.hypot(track["x"] - r.x_common, track["y"] - r.y_common) < radius)).any()
                       for r in rows.itertuples())
            if hits >= 3:
                answer[tracklet_id] = fake_id
                break
    return answer

hits = matched_fake(planted, fakes)
found = planted["tracklets"][planted["tracklets"]["tracklet_id"].isin(hits)]
false_tracks = real["tracklets"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column in zip(axes, ["angle_difference", "length_ratio", "length_spread"]):
    ax.hist(false_tracks[column].clip(upper=false_tracks[column].quantile(0.98)), bins=40, alpha=0.6, density=True, label="real night (false)")
    ax.hist(found[column].clip(upper=false_tracks[column].quantile(0.98)), bins=40, alpha=0.6, density=True, label="planted trails")
    ax.set_title(column)
axes[0].legend(); plt.show()

Planted trails sit near angle difference 0, length ratio 1, spread 0; false tracks are everywhere. The cuts in
`configs/fast.yaml` (≤ 15°, within 1.6×, spread ≤ 0.5, lengths ≤ 32 px) were chosen from tables like these.

## 4. Completeness of the final, consistent tracks

In [ ]:
consistent_ids = set(planted["tracklets"].loc[planted["tracklets"]["consistent"], "tracklet_id"])
recovered = {hits[t] for t in consistent_ids if t in hits}
per_fake = fakes.groupby("injection_id").agg(mag=("mag", "first"), rate=("rate_arcsec_per_hour", "first"))
on_chip = fakes[fakes["x"].between(30, 3040) & fakes["y"].between(30, 3050)].groupby("injection_id").size()
per_fake = per_fake[per_fake.index.isin(on_chip[on_chip >= 3].index)]
per_fake["found"] = per_fake.index.isin(recovered)
print(f"planted trails on the chip in >= 3 frames: {len(per_fake)}, recovered: {per_fake['found'].sum()}")
print(per_fake.groupby(pd.cut(per_fake["mag"], [15, 16, 17, 18, 19, 20]), observed=True)["found"].agg(["mean", "size"]).round(2))
print(f"false fast tracks on the real night: {int(real['tracklets']['consistent'].sum()) if len(real['tracklets']) else 0}")

## Exercises
1. Put 64 back into `streak_lengths_pixels`. What happens to detections and false tracks?
2. Run the search with `standardize_over_night: false`. How many streaks per frame?
3. Which speeds (120 to ~480 "/hr) fall between the point and the fast search? How would you close that gap?